**Motivación del Experimento:** Dado que Isolation Forest obtuvo un rendimiento de detección bajo, el siguiente experimento evalúa **Local Outlier Factor (LOF)**. Debido a su alta complejidad computacional, el algoritmo se aplica bajo restricciones de cómputo.

In [ ]:
# Importar las librerías necesarias y funciones de utilidad
import numpy as np
import pandas as pd
from sklearn.neighbors import LocalOutlierFactor

from src.utils import calculate_age, calculate_distance_km, transform_cyclic_hour

In [ ]:
# Cargar los conjuntos de datos
df_train = pd.read_csv('../data/fraudTrain.csv')
df_test = pd.read_csv('../data/fraudTest.csv')

# Combinar ambos conjuntos de datos en un solo DataFrame
df_raw = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)

# Separar la variable objetivo
y_true = df_raw['is_fraud']
X_raw = df_raw.drop(columns=['is_fraud'])

print("X_raw dimensions:", X_raw.shape)
print("y_true dimensions:", y_true.shape)

X_raw dimensions: (1852394, 22)
y_true dimensions: (1852394,)


In [ ]:
# Aplicar Codificación por Frecuencia usando frecuencias globales
cols_freq = ["category", "job", "state", "merchant"]

cols_to_drop = [
    # Variables codificadas
    "category",
    "job",
    "state",
    "merchant",
    # Variables procesadas
    "trans_date_trans_time",
    "lat",
    "long",
    "merch_lat",
    "merch_long",
    "dob",
    # Variables irrelevantes
    "street",
    "city",
    "zip",
    "unix_time",
    "first",
    "last",
    "cc_num",
    "trans_num",
    "Unnamed: 0",
]

In [ ]:
# Aplicar el mismo pipeline de preprocesamiento utilizado anteriormente
X_raw['trans_date_trans_time'] = pd.to_datetime(X_raw['trans_date_trans_time'])

# Extraer la hora de la transacción
horas = X_raw['trans_date_trans_time'].dt.hour

# Codificar la hora como características cíclicas
(
    X_raw["hour_sin"],
    X_raw["hour_cos"],
) = transform_cyclic_hour(X_raw)

# Generar 'age', 'distance_km' y codificar 'gender'
X_raw["distance_km"] = calculate_distance_km(X_raw)
X_raw["age"] = calculate_age(X_raw)
X_raw['gender'] = X_raw['gender'].map({'F': 0, 'M': 1})

for col in cols_freq:
    # Normalizar frecuencias a valores entre 0 y 1
    freq_map = X_raw[col].value_counts(normalize=True)

    # Crear la variable codificada
    X_raw[f"{col}_freq"] = X_raw[col].map(freq_map)

# Aplicar log1p para reducir el sesgo y comprimir montos
X_raw["amt"] = np.log1p(X_raw["amt"])

# Eliminar columnas innecesarias
X_raw = X_raw.drop(columns=cols_to_drop, errors="ignore")

# Inspeccionar las columnas resultantes
X_raw.columns

Index(['amt', 'gender', 'city_pop', 'hour_sin', 'hour_cos', 'distance_km',
       'age', 'category_freq', 'job_freq', 'state_freq', 'merchant_freq'],
      dtype='str')

In [ ]:
# Configurar y ejecutar el modelo Local Outlier Factor
contamination_rate = (y_true == 1).sum() / len(y_true)

print(f"Exact contamination rate: {contamination_rate:.6f}")

# Inicializar el modelo LOF
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=contamination_rate,
    n_jobs=-1
)

# Ajustar el modelo y predecir anomalías en todo el conjunto de datos
preds_lof_total = lof.fit_predict(X_raw)

# Mapear predicciones:
# -1 → 1 (Anomalía/Fraude)
#  1 → 0 (Normal)
preds_lof_total_binary = [1 if p == -1 else 0 for p in preds_lof_total]

# Evaluar el rendimiento del modelo en la detección de fraude
total_frauds = (y_true == 1).sum()
captured_frauds = sum(1 for p, r in zip(preds_lof_total_binary, y_true) if p == 1 and r == 1)

print(f"Total fraud cases:          {total_frauds:,}")
print(f"Frauds detected by LOF:       {captured_frauds:,}")
print(f"Fraud detection rate (Recall):        {(captured_frauds / total_frauds) * 100:.2f}%")

Exact contamination rate: 0.005210
Total fraud cases:          9,651
Frauds detected by LOF:       2,058
Fraud detection rate (Recall):        21.32%


**Conclusión Final:** En la detección de fraudes financieros con etiquetas históricas, los métodos puramente no supervisados muestran un rendimiento limitado porque identifican anomalías estadísticas en lugar de comportamientos engañosos que simulan intencionalmente transacciones legítimas. Este experimento refuerza por qué los modelos supervisados son el enfoque preferido para sistemas de detección de fraude en entornos de producción.